In [112]:
import pickle
import warnings
import json
warnings.filterwarnings("ignore")

import pandas as pd
import numpy  as np

from sklearn.model_selection    import train_test_split
from pathlib                    import Path
from IPython.display            import display, HTML
from methods.FeatureSelection import (
                                        remove_low_variance_features, 
                                        nan_values, 
                                        const_feature,
                                        fill_missing_values,
                                        get_feature_intervals,
                                        compare_samples
                                    )

In [113]:
np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.precision", 4)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
display(HTML("<style>.container { width:99% !important; }</style>"))
pd.options.display.float_format = '{:,.3f}'.format
pd.set_option('display.max_columns', None)
# ширина каждого столбца (в символах)
pd.set_option('display.max_colwidth', None)

In [114]:
path = f'prepared_data/dataset_for_feature_selection.pickle'

# загрузка данных
with open(path, 'rb') as f:
    samples = pickle.load(f)

# проверим, какие ключи есть в загруженном словаре
print("Ключи в samples:", samples.keys())

Ключи в samples: dict_keys(['df', 'info_fields', 'features', 'cat_features', 'num_features', 'target'])


In [115]:
df           = samples['df']
info         = samples['info_fields']
cat_features = samples['cat_features']
num_features = samples['num_features']
target       = samples['target']

In [116]:
df.shape

(144900, 47)

## Анализ вещественных фичей

In [117]:
feature_reasons = {}
low_var_features                         = remove_low_variance_features(df, num_features, threshold=0.001)
feature_reasons['low_variance_features'] = low_var_features
num_features                             = list(set(num_features) - set(low_var_features))

open                             1,171,145.251
high                             1,200,784.316
low                              1,136,772.051
close                            1,169,620.180
volume             276,312,167,871,055,488.000
ret_1                                    0.000
ret_3                                    0.000
ret_5                                    0.002
ret_20                                   0.010
vol3                                     0.000
vol_5                                    0.000
vol_15                                   0.000
vol_30                                   0.000
vol_45                                   0.000
vol_90                                   0.000
mom_3                                    0.001
mom_5                                    0.002
mom_15                                   0.007
mom_20                                   0.010
sma_3                            1,170,499.306
sma_5                            1,170,301.405
sma_10       

In [118]:
# удаление константных фичей
const_features                    = const_feature(df, num_features,  0.9)
num_features                      = list(set(num_features) - set(const_features))
feature_reasons['const_features'] = const_features

In [119]:
# удаление фичей с пропусками
nan_features                    = nan_values(df, num_features, 0.2)
num_features                    = list(set(num_features) - set(nan_features))
feature_reasons['nan_features'] = nan_features

# кол-во пропусков по данным фичам
if nan_features:
    (df[nan_features].isna().sum() / df.shape[0]).sort_values(ascending=False)

In [120]:
reasons = []
for reason, features in feature_reasons.items():
    for feature, explanation in features.items():
        reasons.append({
            "reason"     : reason,
            "feature"    : feature,
            "explanation": explanation
        })

reasons_df = pd.DataFrame(reasons)

In [121]:
reasons_df

,reason,feature,explanation
0,low_variance_features,ret_1,0.000
1,low_variance_features,ret_3,0.000
2,low_variance_features,vol3,0.000
3,low_variance_features,vol_5,0.000
4,low_variance_features,vol_15,0.000
5,low_variance_features,vol_30,0.000
6,low_variance_features,vol_45,0.000
7,low_variance_features,vol_90,0.000
8,low_variance_features,hl_range,0.000
9,low_variance_features,is_quarter_end,0.000


In [122]:
# очищаем лишние поля
features = num_features + cat_features
# df = df[info + features + [target]]

# Удаляем записи, в которых остались пропуски

In [123]:
df.isna().sum()

date                   0
open                   0
high                   0
low                    0
close                  0
volume                 0
ticker                 0
ret_1                 69
ret_3                 69
ret_5                345
ret_20              1380
vol3                 207
vol_5                345
vol_15              1035
vol_30              2070
vol_45              3105
vol_90              6192
mom_3                207
mom_5                345
mom_15              1035
mom_20              1380
sma_3                138
sma_5                276
sma_10               621
sma_20              1311
sma_ratio           1311
vol_mean_5           276
vol_mean_15          966
vol_mean_45         3036
vol_ratio            276
hl_range               0
dow                    0
month                  0
week                   0
is_month_end           0
is_month_start         0
day_of_month           0
day_of_year            0
quarter                0
is_quarter_end         0


In [124]:
df = df.dropna()

In [125]:
df.isna().sum()

date                0
open                0
high                0
low                 0
close               0
volume              0
ticker              0
ret_1               0
ret_3               0
ret_5               0
ret_20              0
vol3                0
vol_5               0
vol_15              0
vol_30              0
vol_45              0
vol_90              0
mom_3               0
mom_5               0
mom_15              0
mom_20              0
sma_3               0
sma_5               0
sma_10              0
sma_20              0
sma_ratio           0
vol_mean_5          0
vol_mean_15         0
vol_mean_45         0
vol_ratio           0
hl_range            0
dow                 0
month               0
week                0
is_month_end        0
is_month_start      0
day_of_month        0
day_of_year         0
quarter             0
is_quarter_end      0
is_quarter_start    0
is_weekend          0
dow_sin             0
dow_cos             0
month_sin           0
month_cos 

In [126]:
# приводим все категориальные фичи к строковому типу
df[cat_features] = df[cat_features].astype('str')

## Разбиваем на подвыборки

In [127]:
# очищаем лишние поля которые могли оставться в датафрейме послое фичи селекшена
merged_list = info + num_features + cat_features + [target]
result_df   = df[merged_list].copy()
print('Размер: ', result_df.shape)
result_df.head(2)

Размер:  (138708, 33)


,date,ticker,close,month_sin,high,sma_20,day_of_month,low,day_of_year,vol_mean_5,mom_20,ret_5,sma_ratio,sma_3,dow_sin,sma_5,mom_15,vol_ratio,sma_10,open,ret_20,volume,month_cos,vol_mean_15,quarter,mom_5,dow,week,mom_3,dow_cos,month,vol_mean_45,target
90,2014-10-15,AFKS,13.850,-0.866,15.260,16.909,15,13.740,288.000,"19,560,300.000",-0.279,-0.095,-0.137,14.167,0.975,14.080,-0.247,0.638,14.715,15.090,-0.279,"12,481,000.000",0.500,"30,501,106.667",4,-0.095,2.000,42.000,0.041,-0.223,10,"20,261,673.333",0
91,2014-10-16,AFKS,13.300,-0.866,14.140,16.482,16,13.100,289.000,"16,218,400.000",-0.279,-0.089,-0.137,14.050,0.434,13.820,-0.247,0.838,14.635,14.110,-0.279,"13,589,000.000",0.500,"29,109,580.000",4,-0.089,3.000,42.000,-0.026,-0.901,10,"20,414,342.222",1


In [128]:
test_time = result_df[result_df['date']>='2026-01-01']
build = result_df[result_df['date']<'2026-01-01']
build, test  = train_test_split(build, stratify=build[target], test_size=0.2, random_state=42)
train, valid = train_test_split(build, stratify=build[target], test_size=0.2, random_state=42)

In [129]:
# колонки для сравнения compare
features_to_compare = num_features + cat_features

In [130]:
train_valid_comparison_df = compare_samples(train, valid, features_to_compare).reset_index().rename({'index': 'feature'}, axis = 1)[['feature', 'euclidean']].sort_values(by = 'euclidean', ascending = False)
build_test_comparison_df  = compare_samples(build, test, features_to_compare).reset_index().rename({'index': 'feature'}, axis = 1)[['feature', 'euclidean']].sort_values(by = 'euclidean', ascending = False)

In [131]:
# train_valid_comparison_df

In [132]:
drop_features = []

train = train.drop(drop_features, axis = 1, errors = 'ignore')
valid = valid.drop(drop_features, axis = 1, errors = 'ignore')
test  = test.drop(drop_features, axis = 1, errors = 'ignore')
test_time  = test_time.drop(drop_features, axis = 1, errors = 'ignore')

cat_features = [
    col for col in train.drop(info + [target], axis = 1).columns.to_list()
    if col in train.columns and train[col].dtype == "object"
]

In [133]:
train.shape,valid.shape, test.shape, test_time.shape, build.shape

((86901, 33), (21726, 33), (27157, 33), (2924, 33), (108627, 33))

In [134]:
#словарь с ключами:
#      cat_features
#  - unique_list уникальные значение на build выборки
#  - freq_value частовстречаемое значение на build выборки
#  - default_value значение, на которое нужно заменить категорию, которая не вошла в unique_list
# 
#      num_features
#  - min
#  - max
#  - mean



features_intervals = get_feature_intervals(build[num_features + cat_features])
features_intervals

{'close': {'min': np.float64(0.008),
  'max': np.float64(6312.505),
  'mean': np.float64(556.344)},
 'month_sin': {'min': np.float64(-1.0),
  'max': np.float64(1.0),
  'mean': np.float64(-0.039)},
 'high': {'min': np.float64(0.009),
  'max': np.float64(6381.01),
  'mean': np.float64(564.414)},
 'sma_20': {'min': np.float64(0.008),
  'max': np.float64(6329.706),
  'mean': np.float64(555.689)},
 'day_of_month': {'min': np.float64(1.0),
  'max': np.float64(31.0),
  'mean': np.float64(15.981)},
 'low': {'min': np.float64(0.008),
  'max': np.float64(6237.005),
  'mean': np.float64(547.979)},
 'day_of_year': {'min': np.float64(6.0),
  'max': np.float64(362.0),
  'mean': np.float64(189.378)},
 'vol_mean_5': {'min': np.float64(2565.23),
  'max': np.float64(3994494200.0),
  'mean': np.float64(127413804.549)},
 'mom_20': {'min': np.float64(-0.279),
  'max': np.float64(0.331),
  'mean': np.float64(0.008)},
 'ret_5': {'min': np.float64(-0.139),
  'max': np.float64(0.155),
  'mean': np.float64(0.00

In [137]:
with open("models/features_intervals.json", "w", encoding="utf-8") as f:
    json.dump(features_intervals, f, ensure_ascii=False, indent=4)

In [135]:
# пришла пустота на инференс ты меняешь на наиболее вероятное/популярное (для категориальной)  для числовой на среднее
# если числовая пришла выше границы, то замечанием на верзнюю границу если нижнняя на нижнюю

# если пришло значение но его нет в словаре заменяем на other
# как быть если не обучалась на other но пришло на инфуренс категориальное значение которго не было на обучении
# заменять на наиболее вероятное

#  замена новых значений в test 
for feature in features_intervals.keys():
    if feature in cat_features:
        test.loc[test[feature].isna(), feature] = features_intervals[feature]['freq_value']
        test.loc[~ test[feature].isin(features_intervals[feature]['unique_list']), feature] = features_intervals[feature]['default_value']

        test_time.loc[test_time[feature].isna(), feature] = features_intervals[feature]['freq_value']
        test_time.loc[~ test_time[feature].isin(features_intervals[feature]['unique_list']), feature] = features_intervals[feature]['default_value']
    else:
        test.loc[test[feature].isna(), feature] = features_intervals[feature]['mean']
        test.loc[test[feature]>features_intervals[feature]['max'], feature] = features_intervals[feature]['max']
        test.loc[test[feature]<features_intervals[feature]['min'], feature] = features_intervals[feature]['min']

        test_time.loc[test_time[feature].isna(), feature] = features_intervals[feature]['mean']
        test_time.loc[test_time[feature]>features_intervals[feature]['max'], feature] = features_intervals[feature]['max']
        test_time.loc[test_time[feature]<features_intervals[feature]['min'], feature] = features_intervals[feature]['min']

In [136]:
path = f'prepared_data/data_for_modeling.pickle'

samples = {
    'train'       : train,
    'valid'       : valid,
    'test'        : test,
    'test_time'   : test_time,
    'target'      : target,
    'reasons_df'  : reasons_df,
    'info_fields' : info,
    'features'    : train.drop(info + [target], axis = 1).columns.to_list(),
    'cat_features': cat_features,
    'features_intervals' : features_intervals
}
with open(path, 'wb') as f:
    pickle.dump(samples, f)